# Education 360° — Student & Admissions Analysis
Analyze admissions volume, courses, branches, and monthly trends.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_PATH = Path.cwd().parent if Path.cwd().name.lower() == "python" else Path.cwd()
DATA_PATH = PROJECT_PATH / "Dataset"

# Accommodate the source filename spelling found in the project.
FILE_CANDIDATES = {
    "admissions": ["admissions.csv"],
    "branches dim": ["braches dim.csv"],
    "calls": ["calls.csv"],
    "courses dim": ["courses dim.csv"],
    "employees": ["employees.csv"],
    "fees": ["fees.csv"],
    "leads": ["leads.csv"],
    "marketing": ["marketing.csv"],
    "placements": ["placements.csv"],
    "students dim": ["students dim.csv"]
}

def load_table(table_name):
    candidates = FILE_CANDIDATES[table_name]
    for filename in candidates:
        path = DATA_PATH / filename
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(
        f"Could not find {table_name}. Tried: {candidates}. "
        f"Expected dataset folder: {DATA_PATH}"
    )

datasets = {}
for table in FILE_CANDIDATES:
    try:
        datasets[table] = load_table(table)
        print(f"{table:15} {datasets[table].shape}")
    except FileNotFoundError as exc:
        print(exc)

admissions      (2000, 10)
Could not find branches dim. Tried: ['braches_dim.csv']. Expected dataset folder: c:\Users\aadit\Desktop\Data_Analyst_Portfolio\Project 03 — Academy Business Analytics\Dataset
calls           (18226, 7)
courses dim     (5, 3)
Could not find date table. Tried: ['date_table.csv']. Expected dataset folder: c:\Users\aadit\Desktop\Data_Analyst_Portfolio\Project 03 — Academy Business Analytics\Dataset
employees       (30, 5)
fees            (2000, 8)
leads           (2000, 6)
marketing       (150, 8)
placements      (2000, 7)
students dim    (2000, 9)


In [5]:
from pathlib import Path

print("Matching branch files:")

for f in DATA_PATH.iterdir():
    if "branch" in f.name.lower() or "brach" in f.name.lower():
        print(repr(f.name))

Matching branch files:
'braches dim.csv'


In [4]:
admissions = datasets["admissions"].copy()
students = datasets["students dim"].copy()
courses = datasets["courses dim"].copy()
branches = datasets["branches dim"].copy()

for col in ["admission date"]:
    if col in admissions:
        admissions[col] = pd.to_datetime(admissions[col], errors="coerce")
if "admission_date" in students:
    students["admission_date"] = pd.to_datetime(students["admission_date"], errors="coerce")

print("Admissions:", len(admissions))
print("Distinct admitted students:", admissions["student id"].nunique() if "student id" in admissions else "Check column")
display(admissions.head())

KeyError: 'branches dim'

## Admissions by course

In [ ]:
if {"course id","course fee"}.issubset(admissions.columns):
    course_summary = admissions.groupby("course id").agg(
        Admissions=("admission id","nunique"),
        Revenue=("course fee","sum"),
        Average_Fee=("course fee","mean")
    ).reset_index()
    if {"course_id","course_name"}.issubset(courses.columns):
        course_summary = course_summary.merge(
            courses[["course_id","course_name"]],
            left_on="course id", right_on="course_id", how="left"
        )
    display(course_summary.sort_values("Admissions", ascending=False))

## Admissions by branch

In [ ]:
if "branch id" in admissions:
    branch_summary = admissions.groupby("branch id").agg(
        Admissions=("admission id","nunique"),
        Revenue=("course fee","sum")
    ).reset_index()
    if {"branch id","branch name"}.issubset(branches.columns):
        branch_summary = branch_summary.merge(branches, on="branch id", how="left")
    display(branch_summary.sort_values("Admissions", ascending=False))

## Monthly admissions trend

In [ ]:
if "admission date" in admissions:
    admissions["admission_month"] = admissions["admission date"].dt.to_period("M").astype(str)
    monthly = admissions.groupby("admission_month").size().reset_index(name="Admissions")
    display(monthly)
    monthly.plot(x="admission_month", y="Admissions", kind="line", marker="o",
                 figsize=(12,5), title="Monthly Admissions")
    plt.xticks(rotation=60); plt.tight_layout(); plt.show()

## Business questions
- Which course and branch attract the most admissions?
- Are there clear monthly peaks or declines?
- Are 2025 figures partial-year and therefore not directly comparable with full years?